# Structured Output com LangChain, Pydantic e ChatOpenAI

**Disciplina:** [NOME_DA_DISCIPLINA] - GenAI e Advanced Analytics

**Curso:** [NOME_DO_CURSO] - Pos-Graduacao

**Professor(a):** Renan Santos Mendes

**Contato:** renansantosmendes@gmail.com

---

## Objetivos da aula

Nesta aula pratica vamos aprender a obter **structured output** de um LLM utilizando o framework **LangChain**, em conjunto com **Pydantic** para definicao de schemas e o modelo **ChatOpenAI** como motor de geracao.

A aula esta dividida em duas partes:

1. **Exemplo introdutorio**: extracao de dados estruturados a partir de um texto simples, utilizando uma classe Pydantic simples. O objetivo aqui e entender o mecanismo basico antes de aplica-lo em um caso mais complexo.
2. **Estudo de caso completo**: extracao de dados estruturados a partir de curriculos em PDF, utilizando classes Pydantic aninhadas para representar toda a estrutura de um curriculo.

## 1. Configuracao do ambiente

Vamos instalar as bibliotecas necessarias. Utilizamos o `uv` como gerenciador de pacotes por ser significativamente mais rapido que o `pip` tradicional.

In [ ]:
!pip install uv -q
!uv pip install --system langchain langchain-openai pdfplumber pydantic -q


## 2. Importacao das bibliotecas

In [ ]:
from getpass import getpass
from typing import Optional

import pdfplumber
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


## 3. Configuracao da chave de API

A chave de API da OpenAI deve ser inserida de forma segura, sem deixa-la escrita diretamente no notebook.

In [ ]:
import os

openai_api_key = getpass("Insira sua chave de API da OpenAI: ")
os.environ["OPENAI_API_KEY"] = openai_api_key


In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)


---

# Parte 1 - Exemplo introdutorio de structured output

Antes de aplicar structured output em um problema complexo como extracao de curriculos, vamos entender o mecanismo com um exemplo simples: extrair informacoes de uma unica frase de texto livre.

## 4. Definindo uma classe Pydantic simples

O LangChain permite que qualquer classe `BaseModel` do Pydantic seja utilizada como schema de saida estruturada. Cada `Field` pode receber uma `description`, que e enviada ao modelo como instrucao sobre o que aquele campo deve conter. Isso e o que torna a extracao confiavel: o modelo sabe exatamente qual informacao buscar para cada campo.

Vamos comecar com uma classe que representa informacoes basicas de uma pessoa mencionada em um texto.

In [ ]:
class PersonInfo(BaseModel):
    """Basic information about a person mentioned in a text."""

    name: str = Field(description="Full name of the person")
    age: Optional[int] = Field(description="Age of the person, in years, if mentioned")
    profession: Optional[str] = Field(description="Profession or job title of the person, if mentioned")
    city: Optional[str] = Field(description="City where the person lives or works, if mentioned")


## 5. Vinculando o schema ao modelo com `with_structured_output`

O metodo `with_structured_output` do LangChain recebe uma classe Pydantic e retorna uma nova versao do modelo que, ao ser chamada, devolve diretamente uma instancia dessa classe, ja validada, em vez de um texto livre.

In [ ]:
structured_llm = llm.with_structured_output(PersonInfo)


## 6. Testando a extracao em um texto simples

In [ ]:
sample_text = (
    "Marina Alves tem 34 anos e trabalha como engenheira de software em Curitiba. "
    "Ela comecou sua carreira ainda na faculdade, participando de projetos de iniciacao cientifica."
)

extracted_person = structured_llm.invoke(sample_text)
print(extracted_person)


Observe que o resultado ja e um objeto `PersonInfo`, com os campos tipados e validados. Podemos acessar cada atributo diretamente:

In [ ]:
print(extracted_person.name)
print(extracted_person.age)
print(extracted_person.profession)
print(extracted_person.city)


### Testando com um texto que nao contem todas as informacoes

Como todos os campos, exceto `name`, foram definidos como `Optional`, o modelo deve retornar `None` para as informacoes que nao estiverem presentes no texto, em vez de inventar dados.

In [ ]:
incomplete_text = "Pedro Nascimento e um dos fundadores da empresa."

extracted_person_incomplete = structured_llm.invoke(incomplete_text)
print(extracted_person_incomplete)


### Um segundo exemplo: extraindo uma lista de itens

Classes Pydantic tambem podem conter listas, o que permite estruturar textos que mencionam multiplos itens de uma vez. Vamos criar uma classe simples para extrair uma lista de tarefas mencionadas em um texto.

In [ ]:
class TaskList(BaseModel):
    """A list of tasks mentioned in a text."""

    tasks: list[str] = Field(description="List of individual tasks or action items mentioned in the text")
    total_tasks: int = Field(description="Total number of tasks identified in the text")


In [ ]:
structured_task_llm = llm.with_structured_output(TaskList)

tasks_text = (
    "Antes de viajar, preciso terminar o relatorio mensal, enviar o convite da reuniao "
    "de equipe, revisar o contrato com o fornecedor e confirmar a reserva do hotel."
)

extracted_tasks = structured_task_llm.invoke(tasks_text)
print(extracted_tasks)


Com esses dois exemplos, fica claro o padrao geral de uso do structured output com LangChain:

1. Definir uma classe `BaseModel` do Pydantic descrevendo os campos desejados.
2. Chamar `llm.with_structured_output(MinhaClasse)` para obter uma versao estruturada do modelo.
3. Invocar essa versao estruturada com um texto de entrada, recebendo de volta uma instancia validada da classe.

Agora vamos aplicar exatamente o mesmo padrao em um problema mais realista: a extracao de dados de curriculos em PDF.

---

# Parte 2 - Extracao estruturada de dados de curriculos em PDF

## 7. Upload dos curriculos em PDF

Faca o upload dos arquivos de curriculo em PDF que serao utilizados na pratica. No Google Colab, voce pode usar o widget de upload abaixo. Caso esteja rodando localmente, basta colocar os arquivos PDF na mesma pasta do notebook e ajustar os nomes na lista `resume_file_paths`.

In [ ]:
try:
    from google.colab import files

    uploaded_files = files.upload()
    resume_file_paths = list(uploaded_files.keys())
except ImportError:
    resume_file_paths = [
        "ana_beatriz_lima.pdf",
        "carlos_eduardo_ferreira.pdf",
        "juliana_costa_ribeiro.pdf",
    ]

print(resume_file_paths)


## 8. Extracao de texto do PDF

Assim como no exemplo introdutorio, o modelo de linguagem recebe texto como entrada. Portanto, o primeiro passo e converter o conteudo do PDF em texto puro, utilizando a biblioteca `pdfplumber`.

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract raw text content from a PDF file.

    Args:
        pdf_path: Path to the PDF file to be read.

    Returns:
        The concatenated text extracted from all pages of the PDF.
    """
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                full_text += page_text + "\n"
    return full_text


In [ ]:
resume_texts = {
    path: extract_text_from_pdf(path) for path in resume_file_paths
}

print(resume_texts[resume_file_paths[0]])


## 9. Definindo o schema completo do curriculo

Assim como fizemos com `PersonInfo`, agora vamos modelar um curriculo completo. A diferenca e que aqui utilizamos **classes Pydantic aninhadas**: uma classe principal `Resume` que contem listas de outras classes, como `WorkExperience` e `Education`.

Esse e o mesmo padrao do exemplo introdutorio, apenas com um schema mais rico.

In [ ]:
class ContactInfo(BaseModel):
    """Contact information extracted from a resume."""

    full_name: str = Field(description="Full name of the candidate")
    email: Optional[str] = Field(description="Email address of the candidate")
    phone: Optional[str] = Field(description="Phone number of the candidate")
    location: Optional[str] = Field(description="City and state of the candidate")
    linkedin_url: Optional[str] = Field(description="LinkedIn profile URL, if available")


class WorkExperience(BaseModel):
    """A single work experience entry extracted from a resume."""

    job_title: str = Field(description="Job title held by the candidate")
    company_name: str = Field(description="Name of the company")
    start_date: str = Field(description="Start date of the role, as written in the resume")
    end_date: str = Field(description="End date of the role, as written in the resume")
    responsibilities: list[str] = Field(
        description="List of key responsibilities or achievements in this role"
    )


class Education(BaseModel):
    """A single education entry extracted from a resume."""

    degree: str = Field(description="Degree or program name")
    institution: str = Field(description="Name of the educational institution")
    period: str = Field(description="Period of study, as written in the resume")


class Resume(BaseModel):
    """Structured representation of a candidate resume."""

    contact_info: ContactInfo = Field(description="Contact information of the candidate")
    professional_summary: str = Field(description="Short professional summary of the candidate")
    work_experience: list[WorkExperience] = Field(
        description="List of work experiences, ordered from most to least recent"
    )
    education: list[Education] = Field(description="List of academic qualifications")
    technical_skills: list[str] = Field(description="List of technical skills mentioned in the resume")
    years_of_experience_estimate: float = Field(
        description="Estimated total years of professional experience based on the work history"
    )


## 10. Vinculando o schema de curriculo ao modelo

Repetimos exatamente o mesmo padrao utilizado na Parte 1: chamamos `with_structured_output` passando a classe `Resume`.

In [ ]:
structured_resume_llm = llm.with_structured_output(Resume)


## 11. Executando a extracao estruturada nos curriculos

Vamos aplicar o modelo estruturado em todos os curriculos carregados e armazenar os resultados em um dicionario, usando o caminho do arquivo como chave.

In [ ]:
structured_resumes: dict[str, Resume] = {}

for file_path, text_content in resume_texts.items():
    structured_resumes[file_path] = structured_resume_llm.invoke(text_content)

for file_path, resume in structured_resumes.items():
    print(f"Arquivo: {file_path}")
    print(resume.model_dump_json(indent=2))
    print("-" * 80)


Assim como no exemplo introdutorio, o resultado e um objeto Python validado (`Resume`), permitindo acesso direto e seguro aos campos extraidos:

In [ ]:
sample_resume = list(structured_resumes.values())[0]

print(sample_resume.contact_info.full_name)
print(sample_resume.contact_info.email)
print(f"Total de experiencias: {len(sample_resume.work_experience)}")
print(f"Anos de experiencia estimados: {sample_resume.years_of_experience_estimate}")


## 12. Convertendo os resultados em uma tabela

Como os dados agora estao estruturados e validados, e facil transforma-los em um `DataFrame` para analise comparativa entre candidatos.

In [ ]:
import pandas as pd

summary_rows = []
for file_path, resume in structured_resumes.items():
    summary_rows.append(
        {
            "arquivo": file_path,
            "nome": resume.contact_info.full_name,
            "email": resume.contact_info.email,
            "anos_experiencia_estimados": resume.years_of_experience_estimate,
            "quantidade_experiencias": len(resume.work_experience),
            "principais_habilidades": ", ".join(resume.technical_skills[:5]),
        }
    )

summary_dataframe = pd.DataFrame(summary_rows)
summary_dataframe


## 13. Discussao

Alguns pontos importantes para consolidar o aprendizado:

- O metodo `with_structured_output` do LangChain abstrai os detalhes de como o modelo e instruido a retornar dados estruturados, permitindo focar apenas na definicao do schema Pydantic.
- O mesmo padrao (`BaseModel` + `with_structured_output` + `invoke`) funciona tanto para um schema simples de uma frase quanto para um schema aninhado e complexo, como o de um curriculo completo.
- As descricoes (`description`) de cada `Field` sao fundamentais: elas orientam o modelo sobre o que buscar em cada campo, funcionando como uma especie de instrucao embutida no proprio schema.
- Campos `Optional` permitem que o modelo retorne `None` quando a informacao nao estiver presente no texto, evitando que o modelo "invente" dados (alucinacao).

## 14. Exercicios propostos

1. Adicione um novo campo `seniority_level` na classe `Resume`, do tipo `Literal["junior", "pleno", "senior"]`, e observe como o modelo infere esse valor a partir do texto do curriculo.
2. Crie uma nova classe Pydantic simples, `EventInfo`, para extrair data, local e nome de um evento a partir de um texto livre, seguindo o mesmo padrao da Parte 1.
3. Utilize o `summary_dataframe` da secao 12 para ordenar os candidatos pelo numero estimado de anos de experiencia.
4. Experimente trocar o modelo utilizado em `ChatOpenAI` (por exemplo, de `gpt-4o-mini` para `gpt-4o`) e compare a qualidade da extracao nos curriculos com layouts mais incomuns.